FAISS
facebook ai similarity search is a library for efficient similarity search and clusteing of dense vectors . it contains algos that search in sets of vectors of any size , up to ones that possible do not fit in the ram 

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import CharacterTextSplitter

## loading the data
loader = TextLoader("speech.txt")
document = loader.load()


## splitting the text 
text_splitter = CharacterTextSplitter(chunk_size=1000 , chunk_overlap=50)
docs = text_splitter.split_documents(documents=document)

## creating the embeddings 
embeddings = OllamaEmbeddings(model="gemma:2b")

## langchain doesnt just expect any function , it expects an object that follows a small interface 
## it needs two methods - > embed_documents , embed_query 

## so there is a embeddings class and inside it there are two functions which are called methods 

## so i am gonna create a custom embedding class : 
## for gemini , it does not support embeddings like langchain does for open ai 
from langchain_core.embeddings import Embeddings




## creating the embeddings class because this is going to be passed in langchain chroma
## langchain chroma expects a embedding fucntion
class GeminiEmbeddings(Embeddings):

    def __init__(self, client, model_name: str = "gemini-embedding-001"):
        self.client = client
        self.model_name = model_name

    def embed_documents(self, texts):
        # Gemini expects a LIST of texts
        result = self.client.models.embed_content(
            model=self.model_name,
            contents=texts
        )
        # Extract the vector for each document
        return [emb.values for emb in result.embeddings]

    def embed_query(self, text):
        # Gemini expects a list even for 1 text
        result = self.client.models.embed_content(
            model=self.model_name,
            contents=[text]
        )
        # Return only one vector
        return result.embeddings[0].values




Created a chunk of size 1031, which is longer than the specified 1000
/var/folders/rc/gmql9rzj1gv2swvq5fbl262r0000gn/T/ipykernel_32329/545043575.py:16: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model="gemma:2b")
/var/folders/rc/gmql9rzj1gv2swvq5fbl262r0000gn/T/ipykernel_32329/545043575.py:58: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  db = Chroma(


In [4]:

from google.genai import Client
import os
from dotenv import load_dotenv
load_dotenv()

client = Client(api_key=os.environ["GEMINI_API_KEY"])


In [5]:
embedding_function = GeminiEmbeddings(client=client)

In [8]:
from langchain_community.vectorstores import Chroma
texts = [
    "Narendra Modi was born in Vadnagar, Gujarat, and began his political journey as a worker with the Rashtriya Swayamsevak Sangh. Over the years, he rose through the ranks, eventually becoming the Chief Minister of Gujarat and later the Prime Minister of India. His leadership style, economic policies, and international outreach have shaped India's global image in significant ways."
]

db = Chroma(
    embedding_function=embedding_function,
    persist_directory="my_chroma_store"
)

db.add_texts(
    texts=texts,
    ids=["para1"]
)




ClientError: 403 PERMISSION_DENIED. {'error': {'code': 403, 'message': 'Your API key was reported as leaked. Please use another API key.', 'status': 'PERMISSION_DENIED'}}